In [10]:
import os
import sys
import re
from dotenv import load_dotenv
from pathlib import Path
from collections import Counter
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.processors import TemplateProcessing

def get_merge_frequencies(tokenizer_path: Path, corpus_files: list[Path]) -> dict[str, int]:
    """
    Compute approximate frequency of each merge rule by counting 
    how often the merged token appears in the corpus.
    """
    with open(tokenizer_path, "r", encoding="utf-8") as f:
        tok_data = json.load(f)
    
    merges = tok_data["model"]["merges"]  # Can be list-of-lists or list-of-strings
    
    # Read all corpus text
    corpus_text = ""
    for cf in corpus_files:
        with open(cf, "r", encoding="utf-8") as f:
            corpus_text += f.read() + "\n"
    
    merge_freq = {}
    for merge in merges:
        #  Handle modern tokenizers format: [["first", "second"], ...]
        if isinstance(merge, list) and len(merge) == 2:
            first, second = merge
            merged_token = first + second
        # Fallback for older format: "first second"
        elif isinstance(merge, str):
            parts = merge.split(" ")
            if len(parts) == 2:
                merged_token = "".join(parts)
            else:
                continue
        else:
            continue
            
        # Count occurrences of the merged token in corpus
        count = len(re.findall(re.escape(merged_token), corpus_text))
        merge_freq[merged_token] = count  # Use actual token as key for cleaner output
    
    return merge_freq


def compute_compression_ratio(tokenizer: Tokenizer, text_samples: list[str]) -> float:
    """Higher ratio = better compression (fewer tokens per character)"""
    total_chars = sum(len(t) for t in text_samples)
    total_tokens = sum(len(tokenizer.encode(t).ids) for t in text_samples)
    return total_chars / max(total_tokens, 1)


def train_tokenizer_with_analysis(
    corpus_files: list[Path],
    val_texts: list[str],
    vocab_sizes: list[int] = [50, 100, 150, 200, 250, 300],
    output_dir: Path = Path("output/tokenizer_analysis")
) -> dict:
    """
    Train tokenizers at different vocab sizes, compute compression ratios,
    and analyze merge frequencies for the best candidate.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    results = []
    
    print(f"🔍 Testing {len(vocab_sizes)} vocabulary sizes...")
    print("-" * 70)
    
    for vocab_size in vocab_sizes:
        # Train fresh tokenizer
        tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
        tokenizer.pre_tokenizer = ByteLevel()
        tokenizer.decoder = ByteLevelDecoder()
        
        trainer = BpeTrainer(
            vocab_size=vocab_size,
            min_frequency=2,  # Skip ultra-rare merges
            special_tokens=["[PAD]", "[UNK]", "[CLS]", "[EOS]"],
            show_progress=False
        )
        tokenizer.train([str(f) for f in corpus_files], trainer)
        
        # Add post-processor for [EOS]
        eos_id = tokenizer.token_to_id("[EOS]")
        tokenizer.post_processor = TemplateProcessing(
            single="$A [EOS]",
            special_tokens=[("[EOS]", eos_id)]
        )
        
        # Compute metrics
        compression = compute_compression_ratio(tokenizer, val_texts)
        vocab_actual = len(tokenizer.get_vocab())
        
        # Save tokenizer for later inspection
        save_path = output_dir / f"tokenizer_vocab{vocab_size}.json"
        tokenizer.save(str(save_path))
        
        results.append({
            "vocab_size_requested": vocab_size,
            "vocab_size_actual": vocab_actual,
            "compression_ratio": compression,
            "save_path": save_path
        })
        
        print(f"Vocab {vocab_size:3d} → actual {vocab_actual:3d} | "
              f"compression={compression:.3f} | saved: {save_path.name}")
    
    print("-" * 70)
    
    # Find optimal vocab size (elbow method: stop when gain < 0.02 per +50 tokens)
    best_idx = 0
    for i in range(1, len(results)):
        prev_comp = results[i-1]["compression_ratio"]
        curr_comp = results[i]["compression_ratio"]
        vocab_gain = results[i]["vocab_size_actual"] - results[i-1]["vocab_size_actual"]
        
        if vocab_gain > 0:
            comp_gain = curr_comp - prev_comp
            if comp_gain < 0.02:  # Diminishing returns threshold
                best_idx = i - 1
                break
    else:
        best_idx = len(results) - 1  # Use largest if no elbow found
    
    best_result = results[best_idx]
    print(f"\n✅ Recommended vocab size: {best_result['vocab_size_requested']} "
          f"(actual: {best_result['vocab_size_actual']}, compression: {best_result['compression_ratio']:.3f})")
    
    # Analyze merge frequencies for the best tokenizer
    print(f"\n🔎 Analyzing merge frequencies for best tokenizer...")
    merge_freqs = get_merge_frequencies(best_result["save_path"], corpus_files)
    
    # Sort by frequency (descending) and show top/bottom
    sorted_merges = sorted(merge_freqs.items(), key=lambda x: x[1], reverse=True)
    
    print("\n📈 Top 10 most frequent merges:")
    for merge, freq in sorted_merges[:10]:
        print(f"  '{merge}' → {freq:,} occurrences")
    
    print("\n📉 Bottom 10 least frequent merges (your list):")
    for merge, freq in sorted_merges[-10:][::-1]:  # Reverse to show least frequent first
        print(f"  '{merge}' → {freq:,} occurrences")
    
    # Flag merges with very low frequency (potential over-merging)
    low_freq_threshold = 10
    low_freq_merges = [(m, f) for m, f in sorted_merges if f < low_freq_threshold]
    if low_freq_merges:
        print(f"\n⚠️  {len(low_freq_merges)} merges appear < {low_freq_threshold} times in corpus")
        print("   Consider reducing vocab_size or increasing min_frequency")
    
    return {
        "results": results,
        "best_result": best_result,
        "merge_frequencies": merge_freqs,
        "low_freq_merges": low_freq_merges,
        "sorted_merges":sorted_merges
    }


In [11]:
# Define your corpus and validation texts
load_dotenv()
project_root = Path(os.environ.get("PROJECT_ROOT", "."))
corpus_dir = project_root / "data"/"processed"/"tokenizer_corpora"
corpus_files = list(corpus_dir.glob("*.txt"))  # All training texts

# Load a small held-out set for compression evaluation
val_texts = []
for cf in corpus_files[:3]:  # Use first 3 files as pseudo-validation
    with open(cf, "r", encoding="utf-8") as f:
        content = f.read()
        # Split into chunks for evaluation
        chunks = [content[i:i+500] for i in range(0, len(content), 500)]
        val_texts.extend(chunks[:10])  # Take first 10 chunks

# Run analysis
output_path = project_root / "data"/"processed"/"tokenizer"
analysis = train_tokenizer_with_analysis(
    corpus_files=corpus_files,
    val_texts=val_texts,
    vocab_sizes=[50, 100, 150, 200, 250, 300],
    output_dir= output_path/"tokenizer_analysis"
)

🔍 Testing 6 vocabulary sizes...
----------------------------------------------------------------------
Vocab  50 → actual  84 | compression=0.995 | saved: tokenizer_vocab50.json
Vocab 100 → actual 100 | compression=1.237 | saved: tokenizer_vocab100.json
Vocab 150 → actual 150 | compression=1.633 | saved: tokenizer_vocab150.json
Vocab 200 → actual 200 | compression=1.867 | saved: tokenizer_vocab200.json
Vocab 250 → actual 250 | compression=2.001 | saved: tokenizer_vocab250.json
Vocab 300 → actual 300 | compression=2.098 | saved: tokenizer_vocab300.json
----------------------------------------------------------------------

✅ Recommended vocab size: 300 (actual: 300, compression: 2.098)

🔎 Analyzing merge frequencies for best tokenizer...

📈 Top 10 most frequent merges:
  'en' → 75,012 occurrences
  'es' → 62,237 occurrences
  'er' → 54,549 occurrences
  'an' → 53,669 occurrences
  'la' → 51,284 occurrences
  'ra' → 50,030 occurrences
  'on' → 49,585 occurrences
  'qu' → 47,946 occurrenc

In [12]:
analysis["best_result"]

{'vocab_size_requested': 300,
 'vocab_size_actual': 300,
 'compression_ratio': 2.0981955518254303,
 'save_path': PosixPath('/Users/camilabermudezvalderrama/Documents/LMU-STATISTICS & DATA SCIENCE MASTER/SS2026/Thesis/OCC_HTR/data/processed/tokenizer/tokenizer_analysis/tokenizer_vocab300.json')}

In [13]:
analysis["sorted_merges"]

[('en', 75012),
 ('es', 62237),
 ('er', 54549),
 ('an', 53669),
 ('la', 51284),
 ('ra', 50030),
 ('on', 49585),
 ('qu', 47946),
 ('re', 45339),
 ('as', 41969),
 ('ar', 38491),
 ('que', 34584),
 ('or', 34338),
 ('ta', 33155),
 ('el', 32891),
 ('os', 26875),
 ('sa', 26864),
 ('at', 26620),
 ('ma', 24866),
 ('al', 24612),
 ('ia', 24375),
 ('st', 23879),
 ('ca', 23151),
 ('tz', 22072),
 ('na', 21899),
 ('si', 20838),
 ('au', 19094),
 ('ri', 18493),
 ('et', 18370),
 ('ro', 17727),
 ('ay', 17086),
 ('per', 16626),
 ('om', 16573),
 ('is', 16181),
 ('da', 16014),
 ('ant', 15029),
 ('ent', 14159),
 ('ti', 13472),
 ('ot', 13426),
 ('ut', 13395),
 ('eu', 12836),
 ('it', 12721),
 ('us', 12702),
 ('em', 12701),
 ('ys', 12584),
 ('men', 12565),
 ('ol', 11656),
 ('ir', 11596),
 ('va', 10780),
 ('ha', 10387),
 ('ci', 10161),
 ('gu', 9756),
 ('in', 9474),
 ('res', 9395),
 ('eg', 9054),
 ('ran', 8956),
 ('ens', 8626),
 ('il', 8558),
 ('ey', 8217),
 ('ec', 7513),
 ('ic', 7229),
 ('ays', 7094),
 ('atz', 7